# CIDS · 18 — boundary refiner, pilot arm B (frequency, no test)

Refines the **final memory-jury mask M0** (notebook 12, val Dice 0.7742) only within 8 mm of its boundary. Before training the output **is** M0 (checked case by case against notebook 12's `val_metrics.csv`).

The oracle (notebook 16) showed the room: fixing the 8 mm band perfectly would give val Dice 0.950. This pilot asks whether a trained refiner can capture part of it.

**Arm B:** clean jury masks + **error-aware patches**: 25% centred on fixable missed tumour, 25% on fixable extra, 50% on the boundary (training ground truth only).
- Training masks are the clean out-of-fold jury masks (no random roughening: `--p_perturb 0`).
- 24 epochs, validation every 3, **no test evaluation** (`--skip_test`). Test is run once, later, only for the winning setting.

**Settings:** GPU T4 x2 · Internet ON. **Inputs:** cache + notebook 12 output (the jury-mask dataset). **Time:** ~3–5 h.
**Run with Save Version → Save & Run All** (not the draft session), so the outputs are kept.


In [ ]:
REPO_URL            = 'https://github.com/Vemuri-BK/cids.git'
GIT_BRANCH          = 'main'
EXPECTED_GIT_COMMIT = ''      # paste the full commit id you pushed

CACHE_ROOT = '/kaggle/input/datasets/kumarvemuri1/mama-mia-preprocessed-cache-2ch'
S3F_DIR    = '/kaggle/input'   # notebook 12 output is found automatically (folder with summary.json + masks/)
CODE_DIR   = '/kaggle/working/cids'
MODE       = 'freq'
OUT        = f'/kaggle/working/s7_armB_{MODE}'
EPOCHS     = 24
ARM_FLAGS  = '--p_perturb 0 --skip_test --error_sampling'
PATHS = [CACHE_ROOT]

In [ ]:
import os, subprocess
if os.path.isdir(f'{CODE_DIR}/.git'):
    subprocess.run(['git', '-C', CODE_DIR, 'fetch', 'origin', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'pull', 'origin', GIT_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '-b', GIT_BRANCH, REPO_URL, CODE_DIR], check=True)
if EXPECTED_GIT_COMMIT:
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', EXPECTED_GIT_COMMIT], check=True)
head = subprocess.check_output(['git', '-C', CODE_DIR, 'rev-parse', 'HEAD'], text=True).strip()
assert not EXPECTED_GIT_COMMIT or head == EXPECTED_GIT_COMMIT, (head, EXPECTED_GIT_COMMIT)
print('code at commit', head)
%cd {CODE_DIR}

def sh(cmd):
    """Run a shell step, stream its output, and STOP the notebook if it fails (plain !cmd does not).
    Progress bars redraw a line with '\r'; only each line's final state is printed (one line per epoch)."""
    import subprocess
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    buf = b''
    while True:
        chunk = p.stdout.read1(65536)
        if not chunk:
            break
        buf += chunk
        *lines, buf = buf.split(b'\n')
        for line in lines:
            print(line.split(b'\r')[-1].decode('utf-8', 'replace'), flush=True)
    if buf.strip():
        print(buf.split(b'\r')[-1].decode('utf-8', 'replace'), flush=True)
    if p.wait():
        raise RuntimeError(f'step failed (exit {p.returncode}): {cmd}')


In [ ]:
sh(f'pip install -q monai medim einops appdirs')
sh(f'python -m pytest -q tests')

In [ ]:
import torch
from pathlib import Path
assert torch.cuda.is_available(), 'No GPU! Settings -> Accelerator -> GPU T4 x2, then restart.'
for q in PATHS:
    print('OK ' if Path(q).exists() else '!! MISSING', q)
assert all(Path(q).exists() for q in PATHS)
hits = [p.parent for p in Path(S3F_DIR).rglob('summary.json') if (p.parent / 'masks').is_dir() or (p.parent / 'masks.zip').exists()]
print('notebook 12 output:', hits)
assert hits, 'attach the notebook 12 output (Add Input -> Your Work -> Notebooks -> 12_s3f_jury_masks)'
S3F_DIR = str(hits[0])

## Smoke test (~5 min): 8 cases per split, 2 epochs
Check: `epoch 0 (= M0)`, the `patch sampling` line, two epoch lines each followed by a `val edits` line, then `pilot done (test skipped)`.


In [ ]:
sh(f'python scripts/s7_refine.py --mode {MODE} --s3f_dir {S3F_DIR} --cache_root {CACHE_ROOT} --work /kaggle/working/rois_smoke --out /kaggle/working/s7_smoke --limit 8 --epochs 2 --val_every 1 --train_check 4 {ARM_FLAGS}')
sh(f'rm -rf /kaggle/working/s7_smoke /kaggle/working/rois_smoke')


## Full run
**Read:**
- `per-case parity with notebook 12 … max |diff| 0` and `epoch 0 (= M0): val Dice 0.7742`.
- Each validation prints val Dice vs M0 (0.7742), cases better / worse, then **val edits**: voxels added / removed that were helpful vs harmful.
- **train-check**: the same 120 training cases without augmentation. If this rises while val does not, the model is memorising. If neither rises, it is not learning.


In [ ]:
sh(f'python scripts/s7_refine.py --mode {MODE} --s3f_dir {S3F_DIR} --cache_root {CACHE_ROOT} --out {OUT} --epochs {EPOCHS} {ARM_FLAGS}')


In [ ]:
import json, pandas as pd
s = json.load(open(f'{OUT}/pilot_summary.json'))
print(f"best val Dice {s['best_val_dice']:.4f} vs M0 {s['val_dice_m0']:.4f}  ({s['minutes']:.0f} min)")
print(pd.read_csv(f'{OUT}/train_log.csv').to_string())
